In [306]:
import requests
import pandas as pd
import time
import json
import datetime


In [307]:
def OpenAlexAPI(url, params):
    
    # Initialize cursor
    cursor = "*"

    # Create a container
    all_datasets = []
    page = 0
    while cursor:
        params["cursor"] = cursor
        
        # Check for request success
        response = requests.get(url, params=params)
        if response.status_code == 200:
            print(f"Status code: {response.status_code}", ';',  "Page number: ", page)
        if response.status_code != 200:
            print("Oh no! Something went wrong during the live demo! How embarrassing!")
            response.raise_for_status()
        records = response.json()['results']
        
        # create a loop iterator
        record_number = 0
        page += 1
       
       # Loop over all the records
        for record in records:
            record_number += 1
            #print("Page number: ", page, 'Record number:', record_number, record)
            all_datasets.append({'id': record['id'],
                                'doi': record['doi'],
                                'title': record['title'],
                                'type': record['type'],
                                'publication_year': record['publication_year'],
                                'publication_date': record['publication_date'],
                                'primary_topic': record['primary_topic'],
                                'topics': record['topics'][:],
                                'language': record['language'],
                                'retracted': record['is_retracted'],
                                'index location': record['indexed_in'],
                                'primary_location': record['primary_location']['source'],
                                'is_oa': record['open_access']['is_oa'],
                                'oa_status': record['open_access']['oa_status'],
                                'license': record['primary_location']['license'], 
                                'authorships': record['authorships'][:],
                                'affiliations': record['authorships'][0]['affiliations'],
                                'Institution': record['authorships'][0]['institutions'],
                                'apc_list': record['apc_list'],
                                'apc_paid': record['apc_paid'],
                                'fwci': record['fwci'],
                                'has_fulltext': record['has_fulltext'],
                                'cited_by_count': record['cited_by_count'], 
                                'citation_normalized_percentile': record['citation_normalized_percentile'],
                                'cited_by_percentile_year': record['cited_by_percentile_year'],
                                'biblio': record['biblio']
                                })         

        # Update cursor
        cursor = response.json()['meta']['next_cursor']
        
        time.sleep(3)
    return all_datasets

parameters = {
    "per-page": 100,
}   

In [ ]:
Institution_URL = 'https://api.openalex.org/works?page=1&filter=type:dataset,authorships.institutions.lineage:i2800403580|i130238516|i4210115145|i4210120349|i4210159608'
Source_JSONOutFile = str(datetime.datetime.today()).split()[0] + '-UMN-Institutions.json'
with open(Source_JSONOutFile, 'w') as outfile:
    json.dump(OpenAlexAPI(Institution_URL, parameters), outfile)

Status code: 200 ; Page number:  0
Status code: 200 ; Page number:  1
Status code: 200 ; Page number:  2
Status code: 200 ; Page number:  3
Status code: 200 ; Page number:  4


In [ ]:
# Function that creates new dataframe from nested list
def CleanJSON(column,InputDF):
    df = pd.concat([pd.DataFrame(x) for x in InputDF[column]], keys=InputDF.index).reset_index(level=1,drop=True)
    return df

In [ ]:
def missing_index(df):
    missing_indices = ([i for i in range(0, len(df)) if i not in df.index])
    return missing_indices

In [ ]:
pd.set_option("display.max_columns", 55)


In [ ]:
df = pd.read_json(Source_JSONOutFile)
df.head()

In [ ]:
# investigate the columns
df.columns

In [ ]:
# Remove list element from column
df['index location'] = df['index location'].str[0]
df

In [ ]:
# normalize  citation metric columns data

citation_normalized_percentile_df = pd.json_normalize(df['citation_normalized_percentile'])
cited_by_percentile_year_df = pd.json_normalize(df['cited_by_percentile_year'])
biblio_df = pd.json_normalize(df['biblio'])
# concat normalized data into original df

df = pd.concat([df, citation_normalized_percentile_df], axis=1)
df = pd.concat([df, cited_by_percentile_year_df], axis=1)
df = pd.concat([df, biblio_df], axis=1)

#rename column
df = df.rename(columns={'value': 'citation_normalized_percentile-value'})


df.head()

In [ ]:
# normalize columns data
primary_location_df = pd.json_normalize(df['primary_location'])


# extract only display name column
primary_location_df = primary_location_df[['display_name']]

#rename display name column
primary_location_df = primary_location_df.rename(columns={'display_name': 'primary_location-source-display_name'}) 

#primary_location_df = primary_location_df.reindex(range(primary_location_df.index.min(), primary_location_df .index.max()+1))

# concat display name column to main df
df = pd.concat([df, primary_location_df], axis=1)

In [ ]:
# Use function to normalize Authors nested list
authorships_df = CleanJSON('authorships', df)
authorships_df['raw_affiliation_strings'] = authorships_df['raw_affiliation_strings'].str[0]
authorships_df

In [ ]:
missing_index(authorships_df['raw_affiliation_strings'])

In [ ]:
authorships_df.iloc[736]

In [ ]:
df.columns

In [ ]:
df.head()

In [ ]:
# what was , 'min', 'max'
# Create new DataFrame to concat original data from
cleaned_df = df[['id', 'doi', 'type', 'title', 'publication_year', 'publication_date', 'primary_topic', 'topics',
       'language', 'retracted', 'index location', 'is_oa',
       'oa_status', 'license', 'primary_location-source-display_name',
       'apc_list', 'apc_paid', 'fwci', 'has_fulltext','cited_by_count',
       'citation_normalized_percentile-value', 'is_in_top_1_percent',
       'is_in_top_10_percent', 'volume', 'issue', 'first_page',
       'last_page',]].copy()
cleaned_df

In [ ]:
# concat original data to unnested cleaned_df
cleaned_df = pd.concat([authorships_df, cleaned_df], axis=1)
cleaned_df


In [ ]:
cleaned_df.iloc[736]

In [ ]:
# reset index to concat with author information_df
cleaned_df = cleaned_df.reset_index()

# normalize author column
author_info_df= pd.json_normalize(cleaned_df['author'])
author_info_df = author_info_df.rename(columns={'id': 'authorships-author-author_id', 'display_name': 'authorships-author-display_name'})
author_info_df

In [ ]:
# concat author_info_df to cleaned_df
cleaned_df = pd.concat([cleaned_df, author_info_df], axis=1).reset_index(drop=True)
cleaned_df.head()

In [ ]:
# normalize nested Insituttion column
institutions_df = CleanJSON('institutions', cleaned_df)

# remove list format for lineage
institutions_df['lineage'] = institutions_df['lineage'].str[0]
institutions_df

# rename columns
institutions_df = institutions_df.rename(columns={'id': 'institutions-id', 'display_name': 'institutions-display_name', 'type': 'institutions-type'})

In [ ]:
# remove duplicate indices 
institutions_df = institutions_df[~institutions_df.index.duplicated(keep='first')]

# fill nan values on missinging indices
institutions_df = institutions_df.reindex(range(institutions_df.index.min(), institutions_df.index.max()+1))

In [ ]:
missing_index(institutions_df)

In [ ]:
institutions_df.head(50)

In [ ]:
primary_topic_df = CleanJSON('primary_topic', cleaned_df)
primary_topic_df = primary_topic_df.rename(columns={'id': 'primary_topic-id', 'score': 'primary_topic-score', 'display_name': 'primary_topic-display_name', 
                                      'subfield': 'primary_topic-subfield', 'field': 'primary_topic-field', 'domain': 'primary_topic-domain'})
primary_topic_df


In [ ]:
primary_topic_df = primary_topic_df[~primary_topic_df.index.duplicated(keep='first')]

primary_topic_df = primary_topic_df.reindex(range(primary_topic_df.index.min(), primary_topic_df.index.max()+1))


In [ ]:
primary_topic_df

In [ ]:
missing_index(primary_topic_df)

In [ ]:
topics_df = CleanJSON('topics', cleaned_df)
topics_df = topics_df.rename(columns={'id': 'topics-id', 'display_name': 'topics-display_name'})
topics_df

In [ ]:
missing_index(topics_df)

In [ ]:
topics_df = topics_df[~topics_df.index.duplicated(keep='first')]

topics_df = topics_df.reindex(range(topics_df.index.min(), topics_df.index.max()+1))


In [ ]:
missing_index(topics_df)

In [ ]:
subfield_df = pd.json_normalize(topics_df['subfield'])
field_df = pd.json_normalize(topics_df['field'])
domain_df = pd.json_normalize(topics_df['domain'])

# rename 
subfield_df = subfield_df.rename(columns={'id': 'topics-subfield-id', 'display_name': 'topics-subfield-display_name'})
field_df = field_df.rename(columns={'id': 'topics-field-id', 'display_name': 'topics-field-display_name'})
domain_df = domain_df.rename(columns={'id': 'topics-domain-id', 'display_name': 'topics-domain-display_name'})

field_df

In [ ]:
topics_df = topics_df[['topics-id', 'topics-display_name']]

In [ ]:
topics_df=topics_df.reset_index(drop=True)
topics_df = pd.concat([topics_df, subfield_df], axis=1).reset_index(drop=True)
topics_df = pd.concat([topics_df, field_df], axis=1).reset_index(drop=True)
topics_df = pd.concat([topics_df, domain_df], axis=1).reset_index(drop=True)
topics_df

In [ ]:
# concat insitution_df and topics_df
merged_list = [institutions_df, primary_topic_df, topics_df]
merged_df = pd.concat(merged_list, axis=1)
merged_df


In [ ]:
unnested_df = cleaned_df.copy()
unnested_df

In [ ]:
unnested_df.columns

In [ ]:
unnested_df = unnested_df[['author_position', 'countries',
       'is_corresponding', 'raw_author_name', 'raw_affiliation_strings',
       'id', 'doi', 'type', 'title', 'publication_year',
       'publication_date', 'language', 'apc_list', 'apc_paid',
       'fwci', 'has_fulltext', 'cited_by_count',
       'citation_normalized_percentile-value', 'is_in_top_1_percent',
       'is_in_top_10_percent', 'volume', 'issue', 'first_page',
       'last_page', 'retracted', 'index location',
       'is_oa', 'oa_status', 'license', 'primary_location-source-display_name', 'authorships-author-author_id',
       'authorships-author-display_name', 'orcid']]
unnested_df

In [ ]:
unnested_df = pd.concat([unnested_df, merged_df], axis=1)
unnested_df


In [ ]:
# check for nan authors
unnested_df['authorships-author-display_name'].isnull().values.any()

In [ ]:
# remove rows where authors are NaN
#unnested_df = unnested_df.iloc[:10568]
#unnested_df

In [ ]:
unnested_df.columns

In [ ]:
unnested_df = unnested_df[['id', 'doi', 'type', 'institutions-type', 'title', 'publication_year', 'publication_date', 'language', 
       'authorships-author-author_id','authorships-author-display_name', 'author_position', 'orcid',
       'primary_topic-id', 'primary_topic-display_name', 'primary_topic-score', 'primary_topic-subfield', 'primary_topic-field',
       'primary_topic-domain', 'topics-id', 'topics-display_name', 'topics-subfield-id', 'topics-subfield-display_name', 
       'topics-field-id', 'topics-field-display_name', 'topics-domain-id','topics-domain-display_name', 
       'apc_list', 'apc_paid', 'fwci', 'has_fulltext', 'cited_by_count', 'citation_normalized_percentile-value', 
       'is_in_top_1_percent', 'is_in_top_10_percent', 'volume', 'issue', 'first_page', 'last_page', 
       'index location', 'primary_location-source-display_name',
       'retracted', 'index location', 'primary_location-source-display_name', 'is_oa', 'oa_status', 'license',  
       'institutions-id', 'institutions-display_name', 'raw_affiliation_strings',  'country_code', 'ror', 'lineage']]
unnested_df.head()

In [ ]:
unnested_df['authorships-author-display_name'].isnull().values.any()

In [ ]:
unnested_df.head(10)

In [ ]:
unnested_df.to_csv(str(datetime.datetime.today()).split()[0] +'-UMN-Institutions.csv')